In [4]:
!pip install -U datasets huggingface_hub

In [1]:
import re
import math
from collections import Counter
from datasets import load_dataset

In [3]:
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="train"
)

print(dataset)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 36718
})


In [4]:
print("Number of rows:", len(dataset))
print("Columns:", dataset.column_names)

Number of rows: 36718
Columns: ['text']


In [5]:
print(dataset[10]["text"])

 The game 's battle system , the BliTZ system , is carried over directly from Valkyira Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ person . A character can only act once per @-@ turn , but characters can be granted multiple turns at the expense of other characters ' turns . Each character has a field and distance of movement limited by their Action Gauge . Up to nine characters can be assigned to a single mission . During gameplay , characters will call out if something happens to them , such as their health points ( HP ) getting low or being knocked out by enemy attacks . Each character has specific " Potentials " , skills unique to each character . They are divided into " Personal Potential " , which are innate skills that remain unaltered unless otherwise dictated by the story and can either help or impede a character

In [6]:
print(dataset[20]["text"])

In [7]:
text = "\n".join(dataset["text"])

print("Total characters:", len(text))
print("\nFirst 1000 characters:\n")
print(text[:1000])

Total characters: 10929707

First 1000 characters:


 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . 

 The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments 

In [8]:
def clean_text(text):
    # Convert everything to lowercase
    text = text.lower()
    
    # Remove WikiText heading markers
    text = re.sub(r'={1,6}', ' ', text)
    
    # Keep letters, numbers and basic punctuation
    text = re.sub(r"[^a-z0-9.,!?;:'-]+", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


cleaned_text = clean_text(text)

print(cleaned_text[:1000])

valkyria chronicles iii senj no valkyria 3 : unrecorded chronicles japanese : 3 , lit . valkyria of the battlefield 3 , commonly referred to as valkyria chronicles iii outside japan , is a tactical role - playing video game developed by sega and media.vision for the playstation portable . released in january 2011 in japan , it is the third game in the valkyria series . employing the same fusion of tactical and real - time gameplay as its predecessors , the story runs parallel to the first game and follows the nameless , a penal military unit serving the nation of gallia during the second europan war who perform secret black operations and are pitted against the imperial unit calamaty raven . the game began development in 2010 , carrying over a large portion of the work done on valkyria chronicles ii . while it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . character designer raita hon

In [9]:
tokens = re.findall(
    r"\b[a-z0-9]+\b",
    cleaned_text
)

print("First 50 tokens:")
print(tokens[:50])

print("\nTotal number of tokens:", len(tokens))

First 50 tokens:
['valkyria', 'chronicles', 'iii', 'senj', 'no', 'valkyria', '3', 'unrecorded', 'chronicles', 'japanese', '3', 'lit', 'valkyria', 'of', 'the', 'battlefield', '3', 'commonly', 'referred', 'to', 'as', 'valkyria', 'chronicles', 'iii', 'outside', 'japan', 'is', 'a', 'tactical', 'role', 'playing', 'video', 'game', 'developed', 'by', 'sega', 'and', 'media', 'vision', 'for', 'the', 'playstation', 'portable', 'released', 'in', 'january', '2011', 'in', 'japan', 'it']

Total number of tokens: 1753826


In [10]:
for i, token in enumerate(tokens[:20], start=1):
    print(i, "->", token)

1 -> valkyria
2 -> chronicles
3 -> iii
4 -> senj
5 -> no
6 -> valkyria
7 -> 3
8 -> unrecorded
9 -> chronicles
10 -> japanese
11 -> 3
12 -> lit
13 -> valkyria
14 -> of
15 -> the
16 -> battlefield
17 -> 3
18 -> commonly
19 -> referred
20 -> to


In [11]:
unigram_counts = Counter(tokens)

print("Total unique words:", len(unigram_counts))

print("\nMost common 20 words:")
print(unigram_counts.most_common(20))

Total unique words: 64822

Most common 20 words:
[('the', 130771), ('of', 57032), ('and', 50738), ('in', 45026), ('to', 39524), ('a', 36735), ('was', 21008), ('on', 15156), ('s', 15104), ('as', 15063), ('that', 14351), ('for', 13797), ('with', 13012), ('by', 12718), ('is', 11692), ('it', 9278), ('from', 9229), ('at', 9071), ('his', 9020), ('he', 8709)]


In [12]:
words_to_check = [
    "the",
    "of",
    "and",
    "machine",
    "learning"
]

for word in words_to_check:
    print(word, "->", unigram_counts[word])

the -> 130771
of -> 57032
and -> 50738
machine -> 206
learning -> 75


In [13]:
bigram_counts = Counter(
    zip(tokens, tokens[1:])
)

print("Total unique bigrams:", len(bigram_counts))

print("\nMost common 20 bigrams:")
print(bigram_counts.most_common(20))

Total unique bigrams: 706561

Most common 20 bigrams:
[(('of', 'the'), 17355), (('in', 'the'), 11856), (('to', 'the'), 6033), (('on', 'the'), 4523), (('and', 'the'), 4394), (('for', 'the'), 3732), (('at', 'the'), 3198), (('from', 'the'), 3015), (('as', 'a'), 2923), (('by', 'the'), 2920), (('with', 'the'), 2819), (('that', 'the'), 2383), (('as', 'the'), 2341), (('the', 'first'), 2227), (('to', 'be'), 2115), (('in', 'a'), 2112), (('it', 'was'), 2032), (('of', 'a'), 1798), (('during', 'the'), 1536), (('with', 'a'), 1517)]


In [14]:
bigrams_to_check = [
    ("machine", "learning"),
    ("of", "the"),
    ("in", "the"),
    ("artificial", "intelligence")
]

for bigram in bigrams_to_check:
    print(bigram, "->", bigram_counts[bigram])

('machine', 'learning') -> 0
('of', 'the') -> 17355
('in', 'the') -> 11856
('artificial', 'intelligence') -> 26


In [15]:
trigram_counts = Counter(
    zip(
        tokens,
        tokens[1:],
        tokens[2:]
    )
)

print("Total unique trigrams:", len(trigram_counts))

print("\nMost common 20 trigrams:")
print(trigram_counts.most_common(20))

Total unique trigrams: 1372348

Most common 20 trigrams:
[(('one', 'of', 'the'), 869), (('the', 'united', 'states'), 667), (('as', 'well', 'as'), 605), (('part', 'of', 'the'), 534), (('the', 'end', 'of'), 510), (('in', 'the', 'united'), 392), (('end', 'of', 'the'), 364), (('at', 'the', 'time'), 319), (('a', 'number', 'of'), 314), (('known', 'as', 'the'), 263), (('as', 'a', 'result'), 250), (('as', 'part', 'of'), 249), (('due', 'to', 'the'), 238), (('the', 'first', 'time'), 234), (('the', 'rest', 'of'), 223), (('the', 'u', 's'), 216), (('in', 'order', 'to'), 215), (('was', 'the', 'first'), 215), (('such', 'as', 'the'), 214), (('most', 'of', 'the'), 214)]


In [16]:
trigrams_to_check = [
    ("one", "of", "the"),
    ("as", "well", "as"),
    ("machine", "learning", "is")
]

for trigram in trigrams_to_check:
    print(trigram, "->", trigram_counts[trigram])

('one', 'of', 'the') -> 869
('as', 'well', 'as') -> 605
('machine', 'learning', 'is') -> 0


In [17]:
print("========== N-GRAM SUMMARY ==========")

print("Total tokens:", len(tokens))
print("Unique unigrams:", len(unigram_counts))
print("Unique bigrams:", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))

========== N-GRAM SUMMARY ==========
Total tokens: 1753826
Unique unigrams: 64822
Unique bigrams: 706561
Unique trigrams: 1372348


In [18]:
print("========== N-GRAM SUMMARY ==========")

print("Total tokens:", len(tokens))
print("Unique unigrams:", len(unigram_counts))
print("Unique bigrams:", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))

========== N-GRAM SUMMARY ==========
Total tokens: 1753826
Unique unigrams: 64822
Unique bigrams: 706561
Unique trigrams: 1372348


In [19]:
total_words = len(tokens)

unigram_probabilities = {
    word: count / total_words
    for word, count in unigram_counts.items()
}

print("Total words:", total_words)

print("\nExample probabilities:")

for word in ["the", "of", "and", "machine", "learning"]:
    print(word, "->", unigram_probabilities.get(word, 0))

Total words: 1753826

Example probabilities:
the -> 0.0745632691042327
of -> 0.032518619292905906
and -> 0.028929893843516973
machine -> 0.00011745749008168427
learning -> 4.276364930158408e-05


In [20]:
bigram_probabilities = {}

for (word1, word2), count in bigram_counts.items():
    bigram_probabilities[(word1, word2)] = (
        count / unigram_counts[word1]
    )

print(
    "P(learning | machine) =",
    bigram_probabilities.get(
        ("machine", "learning"),
        0
    )
)

P(learning | machine) = 0


In [21]:
trigram_probabilities = {}

for (word1, word2, word3), count in trigram_counts.items():
    trigram_probabilities[(word1, word2, word3)] = (
        count / bigram_counts[(word1, word2)]
    )

print(
    "P(is | machine, learning) =",
    trigram_probabilities.get(
        ("machine", "learning", "is"),
        0
    )
)

P(is | machine, learning) = 0


In [22]:
def get_trigram_predictions(word1, word2, top_n=5):
    
    candidates = []
    
    for (w1, w2, w3), count in trigram_counts.items():
        
        if w1 == word1 and w2 == word2:
            
            probability = (
                count / bigram_counts[(w1, w2)]
            )
            
            candidates.append(
                (w3, probability)
            )
    
    # Sort from highest probability to lowest
    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )
    
    return candidates[:top_n]

In [23]:
predictions = get_trigram_predictions(
    "machine",
    "learning",
    5
)

print(predictions)

[]


In [24]:
def display_predictions(word1, word2, top_n=5):
    
    predictions = get_trigram_predictions(
        word1,
        word2,
        top_n
    )
    
    print(f"Input: {word1} {word2}")
    print("\nTop predicted next words:")
    
    if not predictions:
        print("No prediction found.")
        return
    
    for i, (word, probability) in enumerate(
        predictions,
        start=1
    ):
        print(
            f"{i}. {word} - Probability: {probability:.4f}"
        )

In [25]:
display_predictions(
    "machine",
    "learning",
    5
)

Input: machine learning

Top predicted next words:
No prediction found.


In [26]:
display_predictions(
    "one",
    "of",
    5
)

Input: one of

Top predicted next words:
1. the - Probability: 0.6293
2. his - Probability: 0.0514
3. their - Probability: 0.0196
4. these - Probability: 0.0196
5. which - Probability: 0.0152


In [27]:
display_predictions(
    "in",
    "the",
    5
)

Input: in the

Top predicted next words:
1. united - Probability: 0.0331
2. first - Probability: 0.0162
3. early - Probability: 0.0161
4. area - Probability: 0.0123
5. late - Probability: 0.0119


In [28]:
display_predictions(
    "as",
    "well",
    5
)

Input: as well

Top predicted next words:
1. as - Probability: 0.8368
2. the - Probability: 0.0166
3. in - Probability: 0.0111
4. with - Probability: 0.0069
5. a - Probability: 0.0069


In [29]:
display_predictions(
    "as",
    "well",
    5
)

Input: as well

Top predicted next words:
1. as - Probability: 0.8368
2. the - Probability: 0.0166
3. in - Probability: 0.0111
4. with - Probability: 0.0069
5. a - Probability: 0.0069


In [30]:
sentence = input("Enter a sentence or partial sentence: ")

print("\nYou entered:", sentence)

Enter a sentence or partial sentence:  machine learning



You entered: machine learning


In [31]:
user_words = re.findall(
    r"\b[a-z0-9]+\b",
    sentence.lower()
)

print("Tokens:", user_words)

Tokens: ['machine', 'learning']


In [32]:
if len(user_words) >= 2:
    word1 = user_words[-2]
    word2 = user_words[-1]

    print("Last two words:", word1, word2)
else:
    print("Please enter at least two words.")

Last two words: machine learning


In [33]:
if len(user_words) >= 2:

    word1 = user_words[-2]
    word2 = user_words[-1]

    predictions = get_trigram_predictions(
        word1,
        word2,
        5
    )

    print("\nInput:", sentence)
    print("\nTop predicted next words:")

    if predictions:
        for i, (word, probability) in enumerate(
            predictions,
            start=1
        ):
            print(
                f"{i}. {word} - Probability: {probability:.4f}"
            )
    else:
        print("No prediction found.")


Input: machine learning

Top predicted next words:
No prediction found.


In [34]:
def predict_from_sentence(sentence, top_n=5):

    user_words = re.findall(
        r"\b[a-z0-9]+\b",
        sentence.lower()
    )

    if len(user_words) < 2:
        print("Please enter at least two words.")
        return

    word1 = user_words[-2]
    word2 = user_words[-1]

    predictions = get_trigram_predictions(
        word1,
        word2,
        top_n
    )

    print("\nInput:", sentence)
    print(f"Using previous words: '{word1} {word2}'")
    print("\nTop predicted next words:")

    if not predictions:
        print("No prediction found.")
        return

    for i, (word, probability) in enumerate(
        predictions,
        start=1
    ):
        print(
            f"{i}. {word} - Probability: {probability:.4f}"
        )

In [35]:
predict_from_sentence(
    "machine learning",
    5
)


Input: machine learning
Using previous words: 'machine learning'

Top predicted next words:
No prediction found.


In [36]:
predict_from_sentence(
    "I am learning",
    5
)


Input: I am learning
Using previous words: 'am learning'

Top predicted next words:
No prediction found.


In [37]:
predict_from_sentence(
    "one of the",
    5
)


Input: one of the
Using previous words: 'of the'

Top predicted next words:
1. season - Probability: 0.0120
2. year - Probability: 0.0117
3. first - Probability: 0.0104
4. game - Probability: 0.0098
5. song - Probability: 0.0090


In [38]:
predict_from_sentence(
    "in the",
    5
)


Input: in the
Using previous words: 'in the'

Top predicted next words:
1. united - Probability: 0.0331
2. first - Probability: 0.0162
3. early - Probability: 0.0161
4. area - Probability: 0.0123
5. late - Probability: 0.0119


In [39]:
def get_bigram_predictions(word, top_n=5):

    candidates = []

    for (w1, w2), count in bigram_counts.items():

        if w1 == word:

            probability = count / unigram_counts[w1]

            candidates.append(
                (w2, probability)
            )

    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return candidates[:top_n]

In [40]:
def get_unigram_predictions(top_n=5):

    candidates = []

    for word, count in unigram_counts.items():

        probability = count / total_words

        candidates.append(
            (word, probability)
        )

    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return candidates[:top_n]

In [41]:
def smart_predict(sentence, top_n=5):

    user_words = re.findall(
        r"\b[a-z0-9]+\b",
        sentence.lower()
    )

    if len(user_words) < 1:
        print("Please enter a sentence.")
        return []

    # STEP 1: Try Trigram
    if len(user_words) >= 2:

        word1 = user_words[-2]
        word2 = user_words[-1]

        predictions = get_trigram_predictions(
            word1,
            word2,
            top_n
        )

        if predictions:
            return predictions, "Trigram"

    # STEP 2: Try Bigram
    word = user_words[-1]

    predictions = get_bigram_predictions(
        word,
        top_n
    )

    if predictions:
        return predictions, "Bigram"

    # STEP 3: Try Unigram
    predictions = get_unigram_predictions(
        top_n
    )

    return predictions, "Unigram"

In [42]:
def final_predictor(sentence, top_n=5):

    result = smart_predict(
        sentence,
        top_n
    )

    if not result:
        return

    predictions, model_used = result

    print("=" * 50)
    print("       SMART NEXT-WORD PREDICTOR")
    print("=" * 50)

    print("\nInput sentence:")
    print(sentence)

    print("\nModel used:")
    print(model_used)

    print("\nTop predicted next words:")

    for i, (word, probability) in enumerate(
        predictions,
        start=1
    ):
        print(
            f"{i}. {word:<15} "
            f"Probability: {probability:.4f}"
        )

    print("=" * 50)

In [43]:
final_predictor(
    "machine learning",
    5
)

       SMART NEXT-WORD PREDICTOR

Input sentence:
machine learning

Model used:
Bigram

Top predicted next words:
1. that            Probability: 0.1333
2. curve           Probability: 0.0933
3. the             Probability: 0.0800
4. to              Probability: 0.0800
5. about           Probability: 0.0667


In [44]:
final_predictor(
    "machine learning",
    5
)

       SMART NEXT-WORD PREDICTOR

Input sentence:
machine learning

Model used:
Bigram

Top predicted next words:
1. that            Probability: 0.1333
2. curve           Probability: 0.0933
3. the             Probability: 0.0800
4. to              Probability: 0.0800
5. about           Probability: 0.0667


In [45]:
final_predictor(
    "one of the",
    5
)

       SMART NEXT-WORD PREDICTOR

Input sentence:
one of the

Model used:
Trigram

Top predicted next words:
1. season          Probability: 0.0120
2. year            Probability: 0.0117
3. first           Probability: 0.0104
4. game            Probability: 0.0098
5. song            Probability: 0.0090


In [46]:
final_predictor(
    "in the",
    5
)

       SMART NEXT-WORD PREDICTOR

Input sentence:
in the

Model used:
Trigram

Top predicted next words:
1. united          Probability: 0.0331
2. first           Probability: 0.0162
3. early           Probability: 0.0161
4. area            Probability: 0.0123
5. late            Probability: 0.0119


In [47]:
final_predictor(
    "as well",
    5
)

       SMART NEXT-WORD PREDICTOR

Input sentence:
as well

Model used:
Trigram

Top predicted next words:
1. as              Probability: 0.8368
2. the             Probability: 0.0166
3. in              Probability: 0.0111
4. with            Probability: 0.0069
5. a               Probability: 0.0069


In [48]:
final_predictor(
    "artificial intelligence",
    5
)

       SMART NEXT-WORD PREDICTOR

Input sentence:
artificial intelligence

Model used:
Trigram

Top predicted next words:
1. ai              Probability: 0.1538
2. was             Probability: 0.0769
3. research        Probability: 0.0769
4. the             Probability: 0.0385
5. field           Probability: 0.0385


In [49]:
sentence = input(
    "Enter a sentence or partial sentence: "
)

final_predictor(
    sentence,
    5
)

Enter a sentence or partial sentence:  the united


       SMART NEXT-WORD PREDICTOR

Input sentence:
the united

Model used:
Trigram

Top predicted next words:
1. states          Probability: 0.7623
2. kingdom         Probability: 0.1794
3. nations         Probability: 0.0434
4. arab            Probability: 0.0046
5. confederate     Probability: 0.0011
